In [1]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)
import librosa
import os
import matplotlib.pyplot as plt
import numpy as np
import pickle
import math
import time
import wave
import struct
import random
import gc
import scipy
fea_dir='/workspace/mnt/slave3/tone_training_fea'# fn_fea63.npy

2026-02-06 09:46:33.223457: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-06 09:46:33.256341: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9360] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-06 09:46:33.256368: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-06 09:46:33.256390: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1537] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-06 09:46:33.263250: I tensorflow/core/platform/cpu_feature_g

In [2]:
with open('/workspace/home/cewarman/research/kaldi/CEmix/TAAN/data/train/wav.scp', 'r', encoding='utf8') as f:
    wavlst=["/workspace"+line.strip().split()[1] for line in f.readlines()]

In [3]:
def fea_extractor(file_path):
    y, sr = librosa.load(file_path, sr=16000)
    S_ir = librosa.stft(y=y, n_fft=512, hop_length=160)
    #print(np.abs(S_ir).max())
    ##print(np.abs(S_ir).shape,np.abs(S_ir))
    S_dB = librosa.amplitude_to_db(np.abs(S_ir), ref=np.max)
    #fig, ax = plt.subplots()
    #img = librosa.display.specshow(S_dB, x_axis='time',
    #                     y_axis='mel', sr=sr,
    #                     fmax=16000, ax=ax)
    #fig.colorbar(img, ax=ax, format='%+2.0f dB')
    #ax.set(title='Mel-frequency spectrogram')
    return S_dB.transpose()


In [4]:
fea=fea_extractor(wavlst[0])
print(fea.shape)

(1079, 257)


In [4]:
os.system("mkdir -p {}".format(fea_dir))
for i in range(len(wavlst)):
    fn=os.path.basename(wavlst[i]).split('.')[0]
    
    out_fn="{}/{}_fea63".format(fea_dir,fn)
    if(os.path.isfile(out_fn+'.npy')):
        continue
    else:
        fea=fea_extractor(wavlst[i])[:,1:64]
        np.save(out_fn,fea)
print(i+1,'/',len(wavlst),end='\r')

In [5]:
labs=[]
if(os.path.isfile('data/labels.pkl')):
    labs=pickle.load(open("data/labels.pkl",'rb'))
    print('load labels.pkl done.')
else:
    cmd_out=''
    no_label_list=[]
    with open('data/tmpdir/modify_labs_lst','r') as f:
        lablst=[line.strip() for line in f.readlines()]
    for i in range(len(lablst)):
        print(' '*len(cmd_out),end='\r')
        fn=os.path.basename(lablst[i]).split('.')[0]
        cmd_out="parserlab {}/{} {}".format(i+1,len(lablst),fn)
        print(cmd_out,end='\r')
        try:
            with open('/workspace/home/cewarman/research/ASR/tone_rec_large/data/tmpdir/modify_labs/{}.lab'.format(fn),'r',encoding='utf8') as f:
                lab=[line.strip().split() for line in f.readlines()]
                lab_chunks=[]
                lab_chunk=[]
                for j in range(len(lab)):
                    lab_chunk.append(lab[j])
                    if(j<len(lab)-1):
                        if(len(lab[j+1])==4):
                            lab_chunks.append(lab_chunk)
                            lab_chunk=[]
                    else:
                        lab_chunks.append(lab_chunk)
                
                trn_lab=[]  #trn_lab.append([st,et,t,word])
                for j in range(len(lab_chunks)):
                    #print(lab_chunks[j])
                    t=-1
                    for k in range(len(lab_chunks[j])):
                        if(lab_chunks[j][k][2][:2]=='n_'):
                            t=int(lab_chunks[j][k][2][-3])
                            st=round(float(lab_chunks[j][0][0]),2)
                            et=round(float(lab_chunks[j][-1][1]),2)
                            word=lab_chunks[j][0][-1]
                            #print(st,et,t,word)
                            trn_lab.append([st,et,t,word])
                            break
                    
                labs.append([fn, trn_lab])
            #break
        
        except:
            no_label_list.append(fn)
    
    with open("data/labels.pkl", "wb") as file:
        pickle.dump(labs, file)
    print('output labels.pkl done.')
print(len(labs))

load labels.pkl done.
594532


In [6]:
trainwavlst_path='/workspace/mnt/slave1/data/AIshell3/resplit_trainwavlst'#663256
devwavlst_path='/workspace/mnt/slave1/data/AIshell3/resplit_devwavlst'#74560
testwavlst_path='/workspace/mnt/slave1/data/AIshell3/resplit_testwavlst'#260182
with open(trainwavlst_path,'r',encoding='utf8') as ftr, open(devwavlst_path,'r',encoding='utf8') as fde, open(testwavlst_path,'r',encoding='utf8') as fte:
    trainwavlst=[line.strip().split('/')[-1].split('.')[0] for line in ftr.readlines()]
    devwavlst=[line.strip().split('/')[-1].split('.')[0] for line in fde.readlines()]
    testwavlst=[line.strip().split('/')[-1].split('.')[0] for line in fte.readlines()]
print(len(trainwavlst),len(devwavlst),len(testwavlst))

69033 9953 9049


In [7]:
trainlabs=[]
devlabs=[]
testlabs=[]
total_labs_len=len(labs)
print("total_labs_len =",total_labs_len)
for i in range(total_labs_len):
    print('{}/{}'.format(i+1,total_labs_len),end='                  \r')
    idx=total_labs_len-i-1
    if(labs[idx][0] in testwavlst):
        testlabs.append(labs.pop(idx))
    elif(labs[idx][0] in devwavlst):
        devlabs.append(labs.pop(idx))
    else:
        trainlabs.append(labs.pop(idx))
print(len(trainlabs),len(devlabs),len(testlabs))

total_labs_len = 594532
575530 9953 9049                                                                                                                                                                                                                                                                                                              


In [8]:
#aishell3trnlabs=[]
#trainlabslen=len(trainlabs)
#for i in range(trainlabslen):
#    print('{}/{}'.format(i+1,trainlabslen),end='                  \r')
#    if(trainlabs[i][0] in trainwavlst):
#        aishell3trnlabs.append(trainlabs[i])
#print("\n",len(aishell3trnlabs))

575530/575530                                                                                                                                               
 69033


In [57]:
def get_label_data(label):
    f=np.load(fea_dir+'/{}_fea63.npy'.format(label[0]))[2:]
    #print(label[0],np.shape(f),np.shape(f[-2:-1,:]))
    fw=81
    while len(f)<81:
        f=np.concatenate((f,f[-1:,:]),axis=0)
        #print(np.shape(f))
    half_fw=int((fw-1)/2)
    idx_arr = np.array([[x] for x in range(-half_fw,half_fw+1)])
    #print(idx_arr)
    label_fea=[]
    label_ans=[]
    regions=label[1]
    ff=int(round(float(regions[-1][1])*100))
    ff = ff if ff>=81 else 81
    for i in range(len(regions)):
        sf=int(round(float(regions[i][0])*100))
        ef=int(round(float(regions[i][1])*100))
        cf=int((sf+ef)*0.5)
        lb = cf - half_fw
        rb = cf + half_fw + 1
        #print(sf,cf,ef)
        if(cf >= half_fw and len(f) - cf > half_fw):
            sf = sf - cf + half_fw
            ef = ef - cf + half_fw
        elif(len(f) - cf <= half_fw):
            sf = sf - (len(f) - fw)
            ef = ef - (len(f) - fw)
            rb = len(f)
            lb = rb - fw
        else:
            lb = 0
            rb = fw
            
        #if(half_fw>cf):
        #    lb=0
        #    rb=fw
        #elif(ff-half_fw<=cf):
        #    sf=sf-(ff-fw)
        #    ef=ef-(ff-fw)
        #    rb=ff
        #    lb=int(ff-fw)
        #else:
        #    sf=sf-cf+half_fw
        #    ef=ef-cf+half_fw
        #    lb=int(cf-half_fw)
        #    rb=int(lb+fw)
        identify_arr=np.array([[1 if x >= sf and x <= ef else 0] for x in range(fw)])
        #print(sf,ef,len(identify_arr))
        #print(sf,cf,ef,lb,rb)
        #break
        slice_f=f[lb:rb,:]
        #print(np.shape(idx_arr),np.shape(slice_f),np.shape(identify_arr))
        out_fea=np.concatenate((idx_arr,np.concatenate((identify_arr,slice_f),axis=1)),axis=1)
        #print(np.shape(out_fea))
        #print(out_fea)
        #break
        label_fea.append(out_fea)
        label_ans.append(regions[i][2]-1)
    return label_fea,label_ans
def get_batch_from_labels(labels, batch_size=128, shuffle=True):
    fea_batchs=[]
    ans_batchs=[]
    feas=[]
    anss=[]
    for i in range(len(labels)):
        #print(i,labels[i][0],end='   \r')
        label_fea,label_ans=get_label_data(labels[i])
        #print(len(label_ans),label_ans)
        feas.extend(label_fea)
        anss.extend(label_ans)
        #print(len(feas),len(anss))
        #break
    
    
    if(shuffle):
        npfeas=np.array(feas)
        npanss=np.array(anss)
        p = np.random.permutation(len(npanss))
        ret_feas=npfeas[p]
        ret_anss=npanss[p]
    else:
        ret_feas=np.array(feas)
        ret_anss=np.array(anss)
    
    ret_feas_lst=[]
    ret_anss_lst=[]
    x=0
    while x*batch_size<len(ret_feas):
        if((x+1)*batch_size>len(ret_feas)):
            ret_feas_lst.append(ret_feas[x*batch_size:])
            ret_anss_lst.append(ret_anss[x*batch_size:])
        else:
            ret_feas_lst.append(ret_feas[x*batch_size:(x+1)*batch_size])
            ret_anss_lst.append(ret_anss[x*batch_size:(x+1)*batch_size])
        x=x+1
    return ret_feas_lst, ret_anss_lst
#fea_batch, ans_batch = get_batch_from_labels(trainlabs[:500])
#print(len(fea_batch),len(ans_batch),len(ans_batch[0]),len(ans_batch[-1]))

In [9]:
class data_loader:
    get_data_times=0
    def __init__(self, labels, chunk_size, batch_size=128, shuffle=True):
        self.labels=labels
        self.chunk_size=chunk_size
        self.num_a_round=math.ceil(len(labels)/chunk_size)
        self.shuffle=shuffle
        self.batch_size=batch_size
    def get_batch_data(self):
        if(self.get_data_times%self.num_a_round == 0 and self.shuffle):
            #print(self.get_data_times,'shuffle')
            random.shuffle(self.labels)
        
        if(self.get_data_times%self.num_a_round == self.num_a_round - 1):
            fea_batch, ans_batch = get_batch_from_labels(self.labels[(self.get_data_times%self.num_a_round)*self.chunk_size:], batch_size=self.batch_size, shuffle=self.shuffle)
        else:
            fea_batch, ans_batch = get_batch_from_labels(self.labels[(self.get_data_times%self.num_a_round)*self.chunk_size:((self.get_data_times%self.num_a_round)+1)*self.chunk_size], batch_size=self.batch_size, shuffle=self.shuffle)
        #print(self.get_data_times,self.get_data_times%self.num_a_round,len(fea_batch))
        self.get_data_times=self.get_data_times+1
        return fea_batch, ans_batch
#train_data_loader = data_loader(labs[:round(len(labs)*0.9)], 500)
#dev_data_loader = data_loader(labs[round(len(labs)*0.9):], 500)

In [10]:
def loss(model, x, y, training):
    # training=training is needed only if there are layers with different
    # behavior during training versus inference (e.g. Dropout).
    y_ = model(x, training=training)
    return tf.keras.losses.sparse_categorical_crossentropy(y_true=y, y_pred=y_)
def grad(model, inputs, targets, dropout):
    with tf.GradientTape() as tape:
        loss_value = loss(model, inputs, targets, training=dropout)
    return loss_value, tape.gradient(loss_value, model.trainable_variables)
def my_fit(model, train_labels, dev_labels, epoch_num, batch_size, chunk_size, learning_rate, shuflag=True):
    train_data_loader = data_loader(train_labels, chunk_size, batch_size, shuflag)
    dev_data_loader = data_loader(dev_labels, chunk_size, batch_size, shuflag)
    temp_output=""
    max_dev_acc=0.0
    last_chunk_loss=10000000.0
    best_weight=model.get_weights()
    
    for epoch_idx in range(epoch_num):
        start_seconds = time.time()
        epoch_loss_avg = tf.keras.metrics.Mean()
        epoch_accuracy = tf.keras.metrics.SparseCategoricalAccuracy()
        for chunk_idx in range(train_data_loader.num_a_round):
            fea_batch, ans_batch=train_data_loader.get_batch_data()
            for mini_batch_idx in range(len(ans_batch)):
                loss_value, grads = grad(model, fea_batch[mini_batch_idx], ans_batch[mini_batch_idx], True)
                if(np.isnan(loss_value.numpy()).any()):
                    model.load_weights('models/temp/TAAN_lexicon2/TR_model')
                    model.optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5)
                    continue
                model.optimizer.apply_gradients(zip(grads, model.trainable_variables))
                epoch_loss_avg.update_state(loss_value)
                epoch_accuracy.update_state(ans_batch[mini_batch_idx], model(fea_batch[mini_batch_idx], training=True))
                print(" "*(len(temp_output)+1), end='\r')
                end_seconds = time.time()
                cost_time=round(end_seconds-start_seconds)
                temp_output="{}s, epoch {}, chunk {}/{}, mini_batch {}/{} train_epoch_loss_avg={:.3f} last_chunk_loss={:.3f} train_epoch_accuracy={:.3%}".format(  cost_time,
                                                                                                                                          epoch_idx+1, 
                                                                                                                                          chunk_idx+1, 
                                                                                                                                          train_data_loader.num_a_round, 
                                                                                                                                          mini_batch_idx+1, 
                                                                                                                                          len(ans_batch),
                                                                                                                                          epoch_loss_avg.result(),
                                                                                                                                          last_chunk_loss,
                                                                                                                                          epoch_accuracy.result())
                print(temp_output, end='                                            \r')
            
            if(chunk_idx > 0 and last_chunk_loss > epoch_loss_avg.result()):
                os.system('mkdir -p models/temp/TAAN_lexicon2')
                os.system('cp -r models/temp/TAAN_lexicon/* models/temp/TAAN_lexicon2/')
                last_chunk_loss = epoch_loss_avg.result()
            model.save_weights('models/temp/TAAN_lexicon/TR_model')
            with open('TAAN.log','a') as f:
                f.write("\t{}\n".format(temp_output))
        #########################################################################################################################################################
        dev_loss_avg = tf.keras.metrics.Mean()
        dev_accuracy = tf.keras.metrics.SparseCategoricalAccuracy()
        for chunk_idx in range(dev_data_loader.num_a_round):
            fea_batch, ans_batch=dev_data_loader.get_batch_data()
            for mini_batch_idx in range(len(ans_batch)):
                loss_value, grads = grad(model, fea_batch[mini_batch_idx], ans_batch[mini_batch_idx], False)
                dev_loss_avg.update_state(loss_value)
                dev_accuracy.update_state(ans_batch[mini_batch_idx], model(fea_batch[mini_batch_idx], training=False))
                print(" "*(len(temp_output)+1), end='\r')
                end_seconds = time.time()
                cost_time=round(end_seconds-start_seconds)
                temp_output="{}s, epoch {}, chunk {}/{}, batch {}/{} dev_epoch_loss_avg={:.3f} dev_epoch_accuracy={:.3%}".format(  cost_time,
                                                                                                                                      epoch_idx+1, 
                                                                                                                                      chunk_idx+1, 
                                                                                                                                      dev_data_loader.num_a_round, 
                                                                                                                                      mini_batch_idx+1, 
                                                                                                                                      len(ans_batch),
                                                                                                                                      dev_loss_avg.result(),
                                                                                                                                      dev_accuracy.result())
                print(temp_output,end='\r')
        print(" "*(len(temp_output)+1), end='\r')
        temp_output="Epoch {:03d}: Train Loss: {:.3f}, Train Accuracy: {:.3%}, Dev Loss: {:.3f}, Dev Accuracy: {:.3%}".format(epoch_idx+1,
                                                                                                                    epoch_loss_avg.result(),
                                                                                                                    epoch_accuracy.result(),
                                                                                                                    dev_loss_avg.result(),
                                                                                                                    dev_accuracy.result())
        if(max_dev_acc<dev_accuracy.result()):
            max_dev_acc=dev_accuracy.result()
            model.save_weights('models/TAAN_lexicon/TR_model')
            temp_output=temp_output+"\tsaved."
            best_weight=model.get_weights()
        else:
            model.set_weights(best_weight)
        end_seconds = time.time()
        cost_time=round(end_seconds-start_seconds)
        temp_output="{}s,\t".format(cost_time)+temp_output
        print(temp_output)
        with open('TAAN.log','a') as f:
            f.write("{}\n".format(temp_output))

In [11]:
d_model=256
lindim=512
numhead=2
dt=0.15
x=tf.keras.Input(shape=[81, 65])
xb=tf.keras.layers.BatchNormalization()(x)
ex=tf.keras.layers.Dense(d_model, activation='elu')(xb)
qx=tf.keras.layers.Dense(d_model, activation='elu')(ex)
vx=tf.keras.layers.Dense(d_model, activation='elu')(ex)
kx=tf.keras.layers.Dense(d_model, activation='elu')(ex)
qxb=tf.keras.layers.BatchNormalization()(qx)
vxb=tf.keras.layers.BatchNormalization()(vx)
kxb=tf.keras.layers.BatchNormalization()(kx)
dqx=tf.keras.layers.Dropout(dt)(qxb)
dvx=tf.keras.layers.Dropout(dt)(vxb)
dkx=tf.keras.layers.Dropout(dt)(kxb)
mha = tf.keras.layers.MultiHeadAttention(num_heads=numhead, key_dim=d_model,dropout=dt)(dqx,dkx,dvx)
BAmha=tf.keras.layers.BatchNormalization()(mha+ex)
linear_t = tf.keras.layers.Dense(lindim, activation='elu')(BAmha)
linear = tf.keras.layers.Dense(d_model, activation='elu')(linear_t)
xb2=tf.keras.layers.BatchNormalization()(BAmha+linear)
dxb2=tf.keras.layers.Dropout(dt)(xb2)
qx2=tf.keras.layers.Dense(d_model, activation='elu')(dxb2)
vx2=tf.keras.layers.Dense(d_model, activation='elu')(dxb2)
kx2=tf.keras.layers.Dense(d_model, activation='elu')(dxb2)
qx2b=tf.keras.layers.BatchNormalization()(qx2)
vx2b=tf.keras.layers.BatchNormalization()(vx2)
kx2b=tf.keras.layers.BatchNormalization()(kx2)
dqx2=tf.keras.layers.Dropout(dt)(qx2b)
dvx2=tf.keras.layers.Dropout(dt)(vx2b)
dkx2=tf.keras.layers.Dropout(dt)(kx2b)
mha2 = tf.keras.layers.MultiHeadAttention(num_heads=numhead, key_dim=d_model,dropout=dt)(dqx2,dkx2,dvx2)
BAmha2=tf.keras.layers.BatchNormalization()(mha2+xb2)
linear2_t = tf.keras.layers.Dense(lindim, activation='elu')(BAmha2)
linear2 = tf.keras.layers.Dense(d_model, activation='elu')(linear2_t)
xb3=tf.keras.layers.BatchNormalization()(BAmha2+linear2)
dxb3=tf.keras.layers.Dropout(dt)(xb3)
qx3=tf.keras.layers.Dense(d_model, activation='elu')(dxb3)
vx3=tf.keras.layers.Dense(d_model, activation='elu')(dxb3)
kx3=tf.keras.layers.Dense(d_model, activation='elu')(dxb3)
qx3b=tf.keras.layers.BatchNormalization()(qx3)
vx3b=tf.keras.layers.BatchNormalization()(vx3)
kx3b=tf.keras.layers.BatchNormalization()(kx3)
dqx3=tf.keras.layers.Dropout(dt)(qx3b)
dvx3=tf.keras.layers.Dropout(dt)(vx3b)
dkx3=tf.keras.layers.Dropout(dt)(kx3b)
mha3 = tf.keras.layers.MultiHeadAttention(num_heads=numhead, key_dim=d_model,dropout=dt)(dqx3,dkx3,dvx3)
BAmha3=tf.keras.layers.BatchNormalization()(mha3+xb3)
#linear3 = tf.keras.layers.Dense(lindim, activation='elu')(BAmha3[:,0,:])
#xb4=tf.keras.layers.BatchNormalization()(linear3)
#dxb4=tf.keras.layers.Dropout(0.025)(xb4)
#mid_mha=tf.keras.layers.Flatten()(linear3)
out = tf.keras.layers.Dense(5, activation='softmax')(BAmha3[:,0,:])
model = tf.keras.Model(inputs = x, outputs = out)


2026-02-06 09:48:48.987697: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1883] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9500 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:17:00.0, compute capability: 7.5


In [12]:
model.load_weights('models/temp/TAAN_lexicon/TR_model')
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
my_fit(model=model,train_labels=aishell3trnlabs, dev_labels=devlabs, epoch_num=60,batch_size=256, chunk_size=5000, learning_rate=5e-5)

2026-01-31 01:59:06.747002: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x5f66d87f1c80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-01-31 01:59:06.747048: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2026-01-31 01:59:06.754101: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-01-31 01:59:06.766618: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8907
2026-01-31 01:59:06.822794: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


815s,	Epoch 001: Train Loss: 0.844, Train Accuracy: 67.633%, Dev Loss: 0.641, Dev Accuracy: 76.494%	saved.                                                                    
800s,	Epoch 002: Train Loss: 0.609, Train Accuracy: 78.047%, Dev Loss: 0.566, Dev Accuracy: 79.468%	saved.                                                              
803s,	Epoch 003: Train Loss: 0.546, Train Accuracy: 80.592%, Dev Loss: 0.525, Dev Accuracy: 81.057%	saved.                                                              
804s,	Epoch 004: Train Loss: 0.508, Train Accuracy: 82.106%, Dev Loss: 0.493, Dev Accuracy: 82.356%	saved.                                                              
804s,	Epoch 005: Train Loss: 0.481, Train Accuracy: 83.150%, Dev Loss: 0.485, Dev Accuracy: 82.627%	saved.                                                              
804s,	Epoch 006: Train Loss: 0.460, Train Accuracy: 83.950%, Dev Loss: 0.467, Dev Accuracy: 83.480%	saved.                                           

In [ ]:
my_fit(model=model,train_labels=trainlabs, dev_labels=devlabs, epoch_num=60,batch_size=128, chunk_size=5000, learning_rate=5e-5)

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [12]:
model.load_weights('models/temp/TAAN_lexicon/TR_model')

In [15]:
test_data_loader = data_loader(testlabs, 500, 256, False)

In [16]:
temp_output=''
start_seconds = time.time()
test_loss_avg = tf.keras.metrics.Mean()
test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy()
for chunk_idx in range(test_data_loader.num_a_round):
    fea_batch, ans_batch=test_data_loader.get_batch_data()
    for mini_batch_idx in range(len(ans_batch)):
        loss_value, grads = grad(model, fea_batch[mini_batch_idx], ans_batch[mini_batch_idx], False)
        test_loss_avg.update_state(loss_value)
        test_accuracy.update_state(ans_batch[mini_batch_idx], model(fea_batch[mini_batch_idx], training=False))
        print(" "*(len(temp_output)+1), end='\r')
        end_seconds = time.time()
        cost_time=round(end_seconds-start_seconds)
        temp_output="{:.3f}, chunk {}/{}, batch {}/{} test_epoch_loss_avg={:.3f} test_epoch_accuracy={:.3%}".format(  cost_time,
                                                                                                                              chunk_idx+1, 
                                                                                                                              test_data_loader.num_a_round, 
                                                                                                                              mini_batch_idx+1, 
                                                                                                                              len(ans_batch),
                                                                                                                              test_loss_avg.result(),
                                                                                                                              test_accuracy.result())
        print(temp_output,end='\r')

In [59]:
output_dir='./data/tmpdir/rec_res'
os.system('mkdir -p {}'.format(output_dir))
labidx=0
predresbuffer=np.array([[0,0,0,0,0]])
sub_trainlabs=trainlabs[575000:]
train_data_loader = data_loader(sub_trainlabs, 500, 256, False)
for chunk_idx in range(train_data_loader.num_a_round):
    fea_batch, ans_batch=train_data_loader.get_batch_data()
    for mini_batch_idx in range(len(ans_batch)):
        predres = model.predict(fea_batch[mini_batch_idx],batch_size=256,verbose=0)
        predresbuffer=np.concatenate((predresbuffer, predres), axis=0)
        remainlen=len(predresbuffer)-1
        while len(sub_trainlabs[labidx][1])<=remainlen:
            fn=sub_trainlabs[labidx][0]
            target_predres=predresbuffer[1:len(sub_trainlabs[labidx][1])+1]
            predresbuffer=np.concatenate((np.array([[0,0,0,0,0]]), predresbuffer[len(sub_trainlabs[labidx][1])+1:]), axis=0)
            remainlen=len(predresbuffer)-1
            np.save(output_dir+'/'+sub_trainlabs[labidx][0], target_predres)
            labidx=labidx+1
            if(labidx==len(sub_trainlabs)):
                break
        
    print(chunk_idx+1,'/',train_data_loader.num_a_round,end='           \r')
    if(chunk_idx%50==0):
        gc.collect()
    

In [51]:
gc.collect()

1095701

In [14]:
output_dir='./data/tmpdir/rec_res'
os.system('mkdir -p {}'.format(output_dir))
predreses=[]
for i in range(0,len(trainlabs)):
    print(i+1,'/',len(trainlabs),end='  \r')
    label_fea,label_ans=get_label_data(trainlabs[i])
    #print(label_fea,label_ans)
    predres = model.predict(np.array(label_fea),batch_size=128,verbose=0)
    #gc.collect()
    #print(trainlabs[i][0],predres)
    #print(predres)
    np.save(output_dir+'/'+trainlabs[i][0], predres)
    

In [ ]:
#TB_20160622-011
for i in range(len(trainlabs)):
    if(trainlabs[i][0]=='TB_20160622-011'):
        print(i)
        print(len(trainlabs[i][1]),trainlabs[i])
        for j in range(len(trainlabs[i][1])):
            print(trainlabs[i][1][j])
print(len(trainlabs[0][1]),trainlabs[0])

35927
111 ['TB_20160622-011', [[0.12, 0.46, 2, '所'], [0.46, 0.61, 3, '以'], [0.61, 0.75, 4, '就'], [0.75, 0.89, 4, '是'], [0.89, 1.15, 1, '說'], [1.15, 1.31, 4, '事'], [1.31, 1.37, 2, '實'], [1.37, 1.7, 4, '上'], [1.7, 1.87, 4, '這'], [1.87, 2.21, 4, '是'], [2.24, 2.81, 1, '三'], [3.33, 3.47, 1, '應'], [3.47, 3.65, 1, '該'], [3.7, 3.78, 2, '不'], [3.78, 3.92, 4, '是'], [3.92, 4.08, 1, '說'], [4.08, 4.28, 1, '三'], [4.28, 4.42, 3, '種'], [4.42, 4.52, 4, '不'], [4.52, 4.68, 2, '同'], [4.68, 4.85, 2, '行'], [4.85, 4.91, 4, '業'], [4.91, 5.03, 1, '應'], [5.03, 5.18, 1, '該'], [5.18, 5.42, 1, '說'], [5.69, 6.02, 2, '兩'], [6.02, 6.25, 3, '種'], [6.25, 6.43, 1, '啦'], [6.43, 6.55, 4, '對'], [6.55, 6.61, 2, '不'], [6.61, 6.79, 4, '對'], [6.79, 6.92, 4, '就'], [6.92, 7.04, 4, '是'], [7.07, 7.3, 4, '對'], [7.33, 7.55, 4, '會'], [7.55, 7.74, 4, '計'], [7.74, 7.91, 1, '師'], [7.91, 8.06, 1, '他'], [8.06, 8.29, 1, '真'], [8.29, 8.39, 5, '的'], [8.39, 8.58, 4, '就'], [8.58, 8.89, 4, '是'], [9.06, 9.28, 4, '會'], [9.28, 9.42, 4, '計'], [9.42

In [60]:
rec_res_lst=os.listdir('./data/tmpdir/rec_res')
print(len(rec_res_lst))

575530


In [63]:
all_rec_res=[]
for i in range(len(rec_res_lst)):
    if(rec_res_lst[i][:3]=='SSB' or rec_res_lst[i][-3:]!='npy'):
        continue
    res=np.load('./data/tmpdir/rec_res/'+rec_res_lst[i])
    #print(res.shape)
    pred_ans=res.argmax(axis=1)
    gops=np.zeros(len(pred_ans))
    for j in range(len(res)):
        sorted_indices = np.argsort(res[j])
        first_largest_index = sorted_indices[-1]
        second_largest_index = sorted_indices[-2]
        first_largest_value = res[j][first_largest_index]
        second_largest_value = res[j][second_largest_index]
        gops[j] = np.log(first_largest_value/second_largest_value)
    all_rec_res.append([rec_res_lst[i][:-4], pred_ans, gops])
    #print(all_rec_res,len(all_rec_res[0][1]),len(all_rec_res[0][2]))
    #break
    if(i%1000==0):
        print("{}/{}".format(i,len(rec_res_lst)),end='        \r')

In [64]:
all_GOPs=[]
for i in range(len(all_rec_res)):
    for j in range(len(all_rec_res[i][2])):
        all_GOPs.append(all_rec_res[i][2][j])
    if(i%1000==0):
        print("{}/{}".format(i+1,len(all_rec_res)),end='        \r')

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [65]:
np_all_GOPs=np.array(all_GOPs)
print(np_all_GOPs.shape,np_all_GOPs.mean(),np_all_GOPs.std())
threshold=np_all_GOPs.mean()+2*np_all_GOPs.std()

(35982997,) 4.180478614472417 2.8323994550183618


In [62]:
for i in range(len(all_rec_res)):
    fn=all_rec_res[i][0]
    print(fn)
    predtone=all_rec_res[i][1]
    with open('data/tmpdir/modify_labs/'+fn+'.lab', 'r') as f:
        rawlab=[line.strip().split() for line in f.readlines()]
    idx=0
    for j in range(len(rawlab)):
        if(len(rawlab[j])==4):
            print(rawlab[j][3])
    for j in range(len(rawlab)):
        if(rawlab[j][2][:2]=='n_'):
            rawtone=rawlab[j][2][-3]
            
            print(idx+1,rawtone)
            idx=idx+1
    break

TB_20160622-011
<SILENCE>
所
以
就
是
說
事
實
上
這
是
<SILENCE>
三
<SILENCE>
應
該
<SILENCE>
不
是
說
三
種
不
同
行
業
應
該
說
<SILENCE>
兩
種
啦
對
不
對
就
是
<SILENCE>
對
<SILENCE>
會
計
師
他
真
的
就
是
<SILENCE>
會
計
師
他
有
會
計
師
執
照
<SILENCE>
那
計
帳
士
<SILENCE>
他
就
是
<SILENCE>
記
帳
是
要
取
得
記
帳
是
的
執
照
<SILENCE>
那
另
外
就
是
記
帳
及
報
稅
代
理
人
是
比
較
早
期
來
這
兒
<SILENCE>
公
司
做
稅
務
啦
<SILENCE>
帳
<SILENCE>
務
的
這
些
人
<SILENCE>
對
<SILENCE>
好
那
所
以
我
們
<SILENCE>
1 2
2 3
3 4
4 4
5 1
6 4
7 2
8 4
9 4
10 4
11 1
12 1
13 1
14 2
15 4
16 1
17 1
18 3
19 4
20 2
21 2
22 4
23 1
24 1
25 1
26 2
27 3
28 1
29 4
30 2
31 4
32 4
33 4
34 4
35 4
36 4
37 1
38 1
39 1
40 5
41 4
42 4
43 4
44 4
45 1
46 1
47 3
48 4
49 4
50 1
51 2
52 4
53 4
54 4
55 4
56 4
57 1
58 4
59 4
60 4
61 4
62 4
63 4
64 3
65 2
66 4
67 4
68 4
69 5
70 2
71 4
72 4
73 4
74 4
75 4
76 4
77 4
78 4
79 2
80 4
81 4
82 4
83 3
84 2
85 4
86 3
87 4
88 3
89 2
90 2
91 4
92 1
93 1
94 1
95 4
96 4
97 4
98 1
99 4
100 4
101 5
102 4
103 1
104 2
105 4
106 3
107 4
108 2
109 3
110 3
111 5


In [4]:
###############################################################################
model.save('models/TAAN_fully_model')

INFO:tensorflow:Assets written to: models/TAAN_fully_model/assets


INFO:tensorflow:Assets written to: models/TAAN_fully_model/assets


In [2]:
model = tf.keras.models.load_model('models/TAAN_fully_model/')

In [14]:
fea=np.zeros((81,65))

In [48]:
labs[0]

['BG_20170112-001',
 [[0.0, 0.17, 4, '大'],
  [0.17, 0.35, 4, '自'],
  [0.35, 0.68, 2, '然'],
  [0.68, 1.13, 4, '是'],
  [1.13, 1.25, 3, '我'],
  [1.25, 1.46, 5, '們'],
  [1.46, 1.69, 5, '的'],
  [1.72, 2.0, 4, '教'],
  [2.0, 2.16, 4, '室'],
  [2.16, 2.28, 4, '動'],
  [4.25, 4.54, 3, '手'],
  [4.54, 4.71, 4, '做'],
  [4.93, 5.19, 4, '是'],
  [5.19, 5.28, 3, '我'],
  [5.28, 5.46, 5, '們'],
  [5.46, 5.72, 5, '的'],
  [5.72, 5.96, 2, '學'],
  [6.02, 6.28, 2, '習'],
  [6.28, 6.41, 1, '方'],
  [6.41, 6.47, 4, '式'],
  [8.11, 8.29, 4, '報'],
  [8.32, 8.53, 4, '告'],
  [8.53, 8.75, 1, '分'],
  [8.75, 9.13, 3, '享'],
  [9.13, 9.32, 4, '是'],
  [9.32, 9.44, 3, '我'],
  [9.44, 9.55, 5, '們'],
  [9.55, 9.69, 5, '的'],
  [9.69, 9.89, 2, '學'],
  [9.89, 10.1, 2, '習'],
  [10.1, 10.28, 1, '方'],
  [10.28, 10.34, 3, '法']]]

In [57]:
label_fea,label_ans=get_label_data(['1_1_Male_1_1_A_fan1_PC_1',[[1.41, 1.83, 1, '翻']]])
print(label_fea, np.shape(label_fea[0]))
predrest=model.predict(np.array([label_fea[0]]),verbose=0)
print(predrest)

19 61 81
[array([[-40.        ,   0.        , -31.94056702, ..., -57.96926498,
        -59.70871735, -75.52589417],
       [-39.        ,   0.        , -28.78900909, ..., -60.16187668,
        -64.86436462, -72.01712036],
       [-38.        ,   0.        , -29.13531685, ..., -63.8945961 ,
        -77.581604  , -64.45828247],
       ...,
       [ 38.        ,   0.        , -39.20045471, ..., -74.1765213 ,
        -73.68910217, -67.11407471],
       [ 39.        ,   0.        , -39.33145905, ..., -67.0852356 ,
        -70.74867249, -74.45857239],
       [ 40.        ,   0.        , -37.32284164, ..., -76.00189972,
        -65.90219116, -63.07852554]])] (81, 65)
[[9.2831039e-01 3.8348366e-02 6.0562883e-03 2.7073378e-02 2.1154163e-04]]


In [32]:
wave_fn='../../my-app/local/recv_data/wavs/1_1_Male_1_1_A_fan1_PC_1.wav'
sylboundaries='0.0 1.41 ε\n1.41 1.83 fan\n1.83 2.92 ε\n'

y, sr = librosa.load(wave_fn, sr=16000)
S_ir = librosa.stft(y=y, n_fft=512, hop_length=160)
S_dB = librosa.amplitude_to_db(np.abs(S_ir), ref=np.max)
spec = S_dB.transpose()[:,1:64]

fw=81
half_fw=int((fw-1)/2)
idx_arr = np.array([[x] for x in range(-half_fw,half_fw+1)])

while len(spec)<fw:
    spec=np.concatenate((spec,spec[-1:,:]),axis=0)

syltoks = [syl.split(' ') for syl in sylboundaries.split('\n')[:-1]]
feas=[]
#print(syltoks)
for i in range(len(syltoks)):
    if(syltoks[i][2] == 'ε'):
        continue
    else:
        fea = np.zeros((81,65))
        ct = round((float(syltoks[i][0]) + float(syltoks[i][1])) * 50)
        #print(ct)
        lb = ct - half_fw
        rb = ct + half_fw + 1
        sf=int(round(float(syltoks[i][0])*100))
        ef=int(round(float(syltoks[i][1])*100))
        if(ct >= half_fw and len(spec) - ct > half_fw):
            sf = sf - ct + half_fw
            ef = ef - ct + half_fw
        elif(len(spec) - ct <= half_fw):
            sf = sf - (len(spec) - fw)
            ef = ef - (len(spec) - fw)
            rb = len(spec)
            lb = rb - fw
        else:
            lb = 0
            rb = fw
        identify_arr=np.array([[1 if x >= sf and x <= ef else 0] for x in range(fw)])
        slice_spec=spec[lb:rb,:]
        print(identify_arr.shape,sf,ef)
        fea[:,:1] = idx_arr
        fea[:,1:2] = identify_arr
        fea[:,2:] = slice_spec
        
        

        feas.append(fea)
predrest=model.predict(np.array(feas),verbose=0)
print(predrest)

(81, 1) 19 61
[[9.2831039e-01 3.8348366e-02 6.0562883e-03 2.7073378e-02 2.1154163e-04]]


In [33]:
for i in range(len(feas[0])):
    print(feas[0][i])

[-40.           0.         -31.94056702 -32.90139389 -37.97011948
 -28.6820507  -27.42391014 -46.84263992 -40.30764389 -35.49189377
 -38.2781601  -40.86828613 -60.62416077 -42.34370422 -36.81031799
 -36.50492859 -41.07723236 -36.57779312 -38.86763763 -49.77506256
 -47.87990189 -39.73161316 -40.84362793 -47.29993439 -46.86723328
 -40.26235199 -36.25886154 -39.99403763 -49.42546082 -50.70684814
 -68.96777344 -52.10519409 -53.87352371 -51.35832596 -51.79192734
 -55.44302368 -48.7318573  -50.67295837 -56.66537094 -59.59880829
 -54.12853622 -64.92167664 -58.86769867 -53.97725677 -52.75715637
 -57.41418839 -62.19546509 -53.68983078 -52.76807785 -62.58994293
 -60.49999237 -65.59176636 -58.13206482 -61.05349731 -71.00709534
 -77.57909393 -76.9957428  -70.60932922 -60.51092148 -59.53447723
 -58.11013412 -62.15673065 -57.96926498 -59.70871735 -75.52589417]
[-39.           0.         -28.78900909 -32.54439545 -39.65718842
 -28.6921711  -25.73810768 -33.27548218 -38.24800873 -35.68415833
 -36.0827

In [32]:
fw=81
half_fw=int((fw-1)/2)
idx_arr = np.array([[x] for x in range(-half_fw,half_fw+1)])
identify_arr=np.array([[1 if x >= sf and x <= ef else 0] for x in range(fw)])

NameError: name 'sf' is not defined

In [30]:
fea[:,-1:].shape

(81, 1)